# RemoteState React App Demo

Demonstrates how to run a React app with RemoteState.

To run this notebook while RemoteState is in development, follow the steps below.

1. Checkout code
```bash
git clone https://github.com/bcdev/remotestate
cd remotestate
```

2. Build the TS library
```bash
git clone https://github.com/bcdev/remotestate
cd remotestate/remotestate-ts
npm install
npm run build
```

3. Build the demo app
```bash
cd ../remotestate-demo
npm install
npm run build
```

4. Run this notebook:
```bash
cd ../remotestate-py
pixi shell
jupyter-lab
```

In [9]:
from dataclasses import dataclass
from typing import Literal

import remotestate as rs

Define a store that hold's our UI's data state

In [58]:
ItemStatus = Literal["todo", "ongoing", "done"]


@dataclass
class Item:
    id: int
    created: str
    title: str
    status: ItemStatus


@dataclass
class State:
    items: list[Item]
    selected_id: int | None


Define a service that provides actions and/or queries upon the store

In [66]:
from datetime import datetime


class MyService(rs.Service):
    last_item_id: int = 0

    @rs.action
    def add_item(self, title: str):
        self.last_item_id += 1
        items = self.store.at.items.value
        items.append(Item(
            id=self.last_item_id, 
            created=datetime.now().isoformat().replace("T", " "), 
            title=title, 
            status="todo"
        ))
        self.store.at.items = items

    @rs.action
    def remove_item(self, item_id: int):
        item_index = self.find_item_index(item_id)
        if item_index < 0:
            return
        items = self.store.at.items.value
        items.pop(item_index)
        self.store.at.items = items

    @rs.action
    def update_item_title(self, item_id: int, title: str):
        item_index = self.find_item_index(item_id)
        if item_index < 0:
            return
        self.store.at.items[item_index].title = title

    @rs.action
    def update_item_status(self, item_id: int, status: ItemStatus):
        item_index = self.find_item_index(item_id)
        if item_index < 0:
            return
        self.store.at.items[item_index].status = status

    def find_item_index(self, item_id: int) -> int:
        items = self.store.at.items.value
        for i in range(len(items)):
            if items[i].id == item_id:
                return i
        return -1


In [67]:
store = rs.Store(
    State(
        items=[], 
        selected_id=None
    )
)

service = MyService(store)

In [68]:
store_changes = []

def handle_change(change):
    store_changes.append(change)

store.subscribe(handle_change)

<function remotestate.store.Store.subscribe.<locals>.unsubscribe() -> 'None'>

In [69]:
await service.add_item(title="Wash dishes")
await service.add_item(title="Cancel insurance")

In [70]:
store.state

State(items=[Item(id=1, created='2026-07-03 12:28:05.307910', title='Wash dishes', status='todo'), Item(id=2, created='2026-07-03 12:28:05.308238', title='Cancel insurance', status='todo')], selected_id=None)

In [71]:
await service.update_item_status(1, "ongoing")
await service.update_item_title(2, "Cancel car insurance")

In [72]:
store.state

State(items=[Item(id=1, created='2026-07-03 12:28:05.307910', title='Wash dishes', status='ongoing'), Item(id=2, created='2026-07-03 12:28:05.308238', title='Cancel car insurance', status='todo')], selected_id=None)

In [73]:
await service.update_item_status(1, "done")
await service.remove_item(1)

In [74]:
store.state

State(items=[Item(id=2, created='2026-07-03 12:28:05.308238', title='Cancel car insurance', status='todo')], selected_id=None)

In [75]:
store_changes

[{('items',): [{'id': 1,
    'created': '2026-07-03 12:28:05.307910',
    'title': 'Wash dishes',
    'status': 'todo'}]},
 {('items',): [{'id': 1,
    'created': '2026-07-03 12:28:05.307910',
    'title': 'Wash dishes',
    'status': 'todo'},
   {'id': 2,
    'created': '2026-07-03 12:28:05.308238',
    'title': 'Cancel insurance',
    'status': 'todo'}]},
 {('items', 0, 'status'): 'ongoing'},
 {('items', 1, 'title'): 'Cancel car insurance'},
 {('items', 0, 'status'): 'done'},
 {('items',): [{'id': 2,
    'created': '2026-07-03 12:28:05.308238',
    'title': 'Cancel car insurance',
    'status': 'todo'}]}]

TODO: create a simple UI for the above.

---

Serve the app and display the UI as a side-effect

In [4]:
serve_result = rs.serve(
    # If we wouldn't need actions/queries, we'd have passed rs.Service(store) here
    MyService(store), 
    # The app's build directory or its (dev) URL, e.g., "http://localhost:5173/"
    ui_dist="../../remotestate-demo/dist",
    # The height of the generated UI in CSS pixels
    height=300,
)

We can now change state in the store

In [5]:
store.set("count", 100)

or via 'at'

In [6]:
store.at.count = 200

And get state values from the store

In [7]:
store.get("count")

200

or via 'at' 

In [8]:
store.at.count

200

The serve result carries important info for clients using remotestore

In [9]:
serve_result

Field,Value
Host,localhost
Port,61303
Server URL,http://localhost:61303
WebSocket URL,ws://localhost:61303/ws
UI Base URL,http://localhost:61303
UI URL,http://localhost:61303?t=1783061078&ws=ws%3A%2F%2Flocalhost%3A61303%2Fws


In [12]:
item = Item(id=78, title="Clean dishes", status="todo")
item

Item(id=78, title='Clean dishes', status='todo')

In [15]:
from dataclasses import asdict
asdict(item)


{'id': 78, 'title': 'Clean dishes', 'status': 'todo'}